# PIDA-RL Diabetes — 13. Despliegue de Streamlit con ngrok y GitHub

Este notebook permite visualizar el dashboard de dos maneras:

1. **Temporal:** Streamlit dentro de Google Colab expuesto mediante un túnel de ngrok.
2. **Persistente:** publicación en GitHub y despliegue posterior en Streamlit Community Cloud.

Los secretos ya configurados en Google Colab se leen sin imprimirlos:

- `NGROK_AUTHTOKEN`
- `GIT_USER_NAME`
- `GIT_USER_EMAIL`
- `TOKEN-pida-rl-datos-publicos`

> **Uso responsable:** publicar únicamente código y resultados agregados de simulación. No publicar datos identificables, expedientes, trazas individuales ni tokens.


## 1. Inicialización


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import json
import socket
import time
import py_compile

from google.colab import userdata

PROJECT_NAME = 'pida-rl-datos-publicos'
REPO_URL = 'https://github.com/JssSalas/pida-rl-datos-publicos.git'
PROJECT_ROOT = Path('/content') / PROJECT_NAME

DRIVE_ROOT = next(
    (path for path in [Path('/content/drive'), Path('/content/gdrive')] if (path / 'MyDrive').is_dir()),
    None,
)

if DRIVE_ROOT is None:
    from google.colab import drive
    mount_point = Path('/content/drive')
    if mount_point.exists() and any(mount_point.iterdir()):
        mount_point = Path('/content/gdrive')
    if mount_point.exists() and any(mount_point.iterdir()):
        raise RuntimeError('No hay un punto de montaje vacío para Google Drive. Reinicia la sesión.')
    drive.mount(str(mount_point), force_remount=False)
    DRIVE_ROOT = mount_point

PERSISTENT_ROOT = DRIVE_ROOT / 'MyDrive' / 'PIDA-RL-Diabetes'
DASHBOARD_DIR = PERSISTENT_ROOT / 'dashboard_streamlit'
DASHBOARD_DATA_DIR = PERSISTENT_ROOT / 'dashboard_data'
APP_PATH = DASHBOARD_DIR / 'app.py'
REQUIREMENTS_PATH = DASHBOARD_DIR / 'requirements.txt'

if not PROJECT_ROOT.exists():
    subprocess.check_call(['git', 'clone', REPO_URL, str(PROJECT_ROOT)])
elif not (PROJECT_ROOT / '.git').is_dir():
    raise RuntimeError(f'La ruta no es un repositorio Git: {PROJECT_ROOT}')

required_paths = [APP_PATH, REQUIREMENTS_PATH, DASHBOARD_DATA_DIR]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Faltan archivos o directorios producidos por el notebook 12:\n' + '\n'.join(missing_paths)
    )

os.chdir(PROJECT_ROOT)
print('Aplicación:', APP_PATH)
print('Datos:', DASHBOARD_DATA_DIR)
print('Repositorio:', PROJECT_ROOT)


## 2. Leer y validar secretos

Esta celda comprueba que Colab puede acceder a los secretos. No imprime valores privados. Si Colab solicita autorización, habilita **Notebook access** para este notebook.


In [ ]:
SECRET_NAMES = {
    'ngrok': 'NGROK_AUTHTOKEN',
    'git_user_name': 'GIT_USER_NAME',
    'git_user_email': 'GIT_USER_EMAIL',
    'github_token': 'TOKEN-pida-rl-datos-publicos',
}

def get_required_secret(secret_name):
    try:
        value = userdata.get(secret_name)
    except Exception as error:
        raise RuntimeError(
            f'No fue posible leer el secreto {secret_name}. '
            'Verifica que exista y que Notebook access esté habilitado. '
            f'Detalle: {error}'
        ) from error

    if not value:
        raise ValueError(
            f'El secreto {secret_name} está vacío o no tiene acceso habilitado para este notebook.'
        )
    return value

ngrok_authtoken = get_required_secret(SECRET_NAMES['ngrok'])
git_user_name = get_required_secret(SECRET_NAMES['git_user_name'])
git_user_email = get_required_secret(SECRET_NAMES['git_user_email'])
github_token = get_required_secret(SECRET_NAMES['github_token'])

print('Secretos de ngrok, identidad Git y token de GitHub disponibles para la sesión.')


## 3. Configurar identidad Git local

La identidad se configura solamente en el clon de `/content/pida-rl-datos-publicos`. No cambia la configuración global de Git y no revela los valores de los secretos.


In [ ]:
os.chdir(PROJECT_ROOT)

subprocess.run(
    ['git', 'config', '--local', 'user.name', git_user_name],
    check=True,
)
subprocess.run(
    ['git', 'config', '--local', 'user.email', git_user_email],
    check=True,
)

configured_name = subprocess.run(
    ['git', 'config', '--local', '--get', 'user.name'],
    text=True,
    capture_output=True,
    check=True,
).stdout.strip()
configured_email = subprocess.run(
    ['git', 'config', '--local', '--get', 'user.email'],
    text=True,
    capture_output=True,
    check=True,
).stdout.strip()

if configured_name != git_user_name or configured_email != git_user_email:
    raise RuntimeError('La identidad Git local no coincide con los secretos configurados.')

print('Identidad Git local configurada correctamente.')


# Versión A — ngrok

La URL de ngrok será temporal. El dashboard dejará de estar disponible al cerrar el túnel, detener Streamlit o reiniciar el entorno de Colab.


## A1. Instalar dependencias


In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQUIREMENTS_PATH)],
    check=True,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'pyngrok'],
    check=True,
)

print('Dependencias de Streamlit, Plotly y pyngrok instaladas.')


## A2. Validar e iniciar dashboard

Primero se valida que `app.py` compile. Después se inicia Streamlit y se abre un túnel HTTP hacia el puerto 8501.


In [ ]:
py_compile.compile(str(APP_PATH), doraise=True)

from pyngrok import ngrok

ngrok.set_auth_token(ngrok_authtoken)

PORT = 8501

def port_is_open(host, port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(1)
        return sock.connect_ex((host, port)) == 0

if 'streamlit_process' in globals() and streamlit_process.poll() is None:
    streamlit_process.terminate()
    streamlit_process.wait(timeout=10)

if 'public_tunnel' in globals():
    try:
        ngrok.disconnect(public_tunnel.public_url)
    except Exception:
        pass

streamlit_process = subprocess.Popen(
    [
        sys.executable,
        '-m',
        'streamlit',
        'run',
        str(APP_PATH),
        '--server.port',
        str(PORT),
        '--server.address',
        '0.0.0.0',
        '--server.headless',
        'true',
        '--browser.gatherUsageStats',
        'false',
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)

for _ in range(30):
    if port_is_open('127.0.0.1', PORT):
        break
    time.sleep(1)
else:
    raise RuntimeError('Streamlit no abrió el puerto 8501 en el tiempo esperado.')

public_tunnel = ngrok.connect(addr=PORT, proto='http')

print('Dashboard activo dentro de Colab: http://127.0.0.1:8501')
print('Abre esta URL temporal de ngrok:', public_tunnel.public_url)


## A3. Cerrar la vista temporal


In [ ]:
if 'public_tunnel' in globals():
    try:
        ngrok.disconnect(public_tunnel.public_url)
        print('Túnel de ngrok cerrado.')
    except Exception as error:
        print('No fue posible cerrar el túnel o ya estaba cerrado:', error)

if 'streamlit_process' in globals() and streamlit_process.poll() is None:
    streamlit_process.terminate()
    streamlit_process.wait(timeout=10)
    print('Proceso de Streamlit detenido.')


# Versión B — GitHub y Streamlit Community Cloud

Esta sección copia solo el código del dashboard y los resultados agregados a la copia local del repositorio. La publicación requiere confirmación explícita mediante el valor `PUBLICAR`.


## B1. Copiar archivos permitidos al repositorio


In [ ]:
REPO_APP_DIR = PROJECT_ROOT / 'dashboard_streamlit'
REPO_DATA_DIR = PROJECT_ROOT / 'dashboard_data'

shutil.copytree(DASHBOARD_DIR, REPO_APP_DIR, dirs_exist_ok=True)
shutil.copytree(DASHBOARD_DATA_DIR, REPO_DATA_DIR, dirs_exist_ok=True)

allowed_data_files = [
    'tabla_decision_final_v1.csv',
    'cobertura_por_riesgo_dashboard_v1.csv',
    'alto_riesgo_por_semilla_dashboard_v1.csv',
    'conclusiones_dashboard_v1.csv',
    'configuracion_dashboard_v1.json',
]

required_repo_files = [
    REPO_APP_DIR / 'app.py',
    REPO_APP_DIR / 'requirements.txt',
    REPO_APP_DIR / 'README.md',
] + [REPO_DATA_DIR / name for name in allowed_data_files]

missing_repo_files = [str(path.relative_to(PROJECT_ROOT)) for path in required_repo_files if not path.is_file()]
if missing_repo_files:
    raise FileNotFoundError('Faltan archivos tras la copia al repositorio:\n' + '\n'.join(missing_repo_files))

print('Archivos permitidos preparados para GitHub:')
for path in required_repo_files:
    print('-', path.relative_to(PROJECT_ROOT), '| bytes:', path.stat().st_size)


## B2. Revisar cambios

Revisa esta salida antes de publicar. Deben aparecer solamente `dashboard_streamlit/` y `dashboard_data/`, con archivos agregados y sin información sensible.


In [ ]:
os.chdir(PROJECT_ROOT)

status = subprocess.run(
    ['git', 'status', '--branch', '--short'],
    text=True,
    capture_output=True,
    check=True,
)

print(status.stdout if status.stdout.strip() else 'No hay cambios pendientes.')

tracked_dashboard_files = subprocess.run(
    ['git', 'ls-files', 'dashboard_streamlit', 'dashboard_data'],
    text=True,
    capture_output=True,
    check=True,
)

if tracked_dashboard_files.stdout.strip():
    print('\nArchivos ya versionados del dashboard:')
    print(tracked_dashboard_files.stdout.strip())


## B3. Corregir autor del último commit local, si aplica

Esta celda es opcional. Solo corrige la autoría si el último commit local fue creado con `Your Name <you@example.com>` y aún no fue enviado a GitHub. No cambia los archivos del dashboard.


In [ ]:
os.chdir(PROJECT_ROOT)

last_commit = subprocess.run(
    ['git', 'log', '-1', '--format=%H|%an|%ae|%s'],
    text=True,
    capture_output=True,
    check=True,
).stdout.strip()

commit_hash, author_name, author_email, subject = last_commit.split('|', 3)

ahead_count = subprocess.run(
    ['git', 'rev-list', '--count', 'origin/main..HEAD'],
    text=True,
    capture_output=True,
    check=True,
).stdout.strip()

should_amend = (
    ahead_count != '0'
    and author_name == 'Your Name'
    and author_email == 'you@example.com'
)

if should_amend:
    subprocess.run(
        ['git', 'commit', '--amend', '--reset-author', '--no-edit'],
        check=True,
)
    print('Autor del último commit local corregido con los secretos de Colab.')
else:
    print('No se requiere corregir la autoría del último commit.')


## B4. Confirmar y publicar en GitHub

Escribe exactamente `PUBLICAR` para autorizar un `git add`, `git commit` si hay cambios y `git push` a la rama actual. La celda no imprime el token.


In [ ]:
CONFIRMACION_PUBLICACION = 'NO'

if CONFIRMACION_PUBLICACION != 'PUBLICAR':
    print('No se realizó commit ni push. Cambia CONFIRMACION_PUBLICACION a PUBLICAR solo después de revisar B2.')
else:
    os.chdir(PROJECT_ROOT)

    subprocess.run(
        ['git', 'add', 'dashboard_streamlit', 'dashboard_data'],
        check=True,
)

    staged = subprocess.run(
        ['git', 'diff', '--cached', '--name-only'],
        text=True,
        capture_output=True,
        check=True,
).stdout.strip()

    if staged:
        print('Archivos preparados para commit:')
        print(staged)

        commit = subprocess.run(
            ['git', 'commit', '-m', 'Agregar dashboard Streamlit y resultados agregados de simulación'],
            text=True,
            capture_output=True,
)

        if commit.returncode != 0:
            raise RuntimeError(
                'El commit falló. Mensaje de Git:\n'
                + (commit.stdout or '')
                + (commit.stderr or '')
            )

        print(commit.stdout.strip())
    else:
        print('No hay cambios nuevos para crear un commit.')

    branch = subprocess.run(
        ['git', 'branch', '--show-current'],
        text=True,
        capture_output=True,
        check=True,
).stdout.strip()

    if not branch:
        raise RuntimeError('No se detectó una rama activa. Ejecuta git switch main antes de publicar.')

    token_remote = (
        'https://x-access-token:'
        + github_token
        + '@github.com/JssSalas/pida-rl-datos-publicos.git'
)

    push = subprocess.run(
        ['git', 'push', token_remote, branch],
        text=True,
        capture_output=True,
)

    if push.returncode != 0:
        raise RuntimeError(
            'El push falló. Verifica que el token fine-grained tenga acceso al repositorio '
            'y permiso Contents: Read and write. Mensaje de Git:\n'
            + (push.stdout or '')
            + (push.stderr or '')
        )

    print(push.stdout.strip())
    print('Publicación completada en GitHub, rama:', branch)


## B5. Verificación posterior al push


In [ ]:
os.chdir(PROJECT_ROOT)

status = subprocess.run(
    ['git', 'status', '--branch', '--short'],
    text=True,
    capture_output=True,
    check=True,
)
print(status.stdout if status.stdout.strip() else 'Estado limpio.')

files_in_head = subprocess.run(
    ['git', 'ls-tree', '-r', '--name-only', 'HEAD'],
    text=True,
    capture_output=True,
    check=True,
).stdout.splitlines()

expected_github_paths = [
    'dashboard_streamlit/app.py',
    'dashboard_streamlit/requirements.txt',
    'dashboard_data/tabla_decision_final_v1.csv',
    'dashboard_data/cobertura_por_riesgo_dashboard_v1.csv',
    'dashboard_data/alto_riesgo_por_semilla_dashboard_v1.csv',
    'dashboard_data/conclusiones_dashboard_v1.csv',
    'dashboard_data/configuracion_dashboard_v1.json',
]

missing_in_head = [path for path in expected_github_paths if path not in files_in_head]
if missing_in_head:
    raise FileNotFoundError('Faltan archivos en el último commit local:\n' + '\n'.join(missing_in_head))

print('Verificación local aprobada. Archivos requeridos presentes en HEAD.')


## B6. Desplegar en Streamlit Community Cloud

Después de ejecutar B4 exitosamente:

1. Abre [Streamlit Community Cloud](https://share.streamlit.io/).
2. Recarga la página o vuelve a crear la aplicación.
3. Selecciona el repositorio `JssSalas/pida-rl-datos-publicos`.
4. Selecciona la rama `main`.
5. En **Main file path**, escribe:

```text
dashboard_streamlit/app.py
```

6. Selecciona **Deploy**.

La aplicación leerá los archivos en `dashboard_data/` porque esa carpeta queda al mismo nivel que `dashboard_streamlit/` dentro del repositorio.


## Resultado esperado

- A2 imprime una URL temporal de ngrok para visualizar Streamlit desde Colab.
- B4 publica las carpetas `dashboard_streamlit/` y `dashboard_data/` después de confirmar `PUBLICAR`.
- B6 despliega la aplicación persistente usando `dashboard_streamlit/app.py`.
